# 02 — EDA: Churn Drivers

Run **after** `dbt build`. Reads `analytics.mart_user_churn_features`.

Produces summary tables and plots for the headline churn drivers used in the case study.

In [ ]:
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
DUCKDB_PATH = Path('../data/processed/kkbox.duckdb')
con = duckdb.connect(str(DUCKDB_PATH), read_only=True)

In [ ]:
df = con.execute('SELECT * FROM analytics.mart_user_churn_features').df()
print('Shape:', df.shape)
df.head()

## Headline churn rate

In [ ]:
overall_churn = df['is_churn'].mean()
print(f'Overall churn rate: {overall_churn:.2%}')
print(f'Churned users: {df["is_churn"].sum():,}')
print(f'Retained users: {(df["is_churn"] == 0).sum():,}')

## Churn by auto-renew status

In [ ]:
auto_renew_churn = df.groupby('latest_is_auto_renew')['is_churn'].agg(['count', 'mean']).rename(columns={'mean': 'churn_rate'})
auto_renew_churn

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
auto_renew_churn['churn_rate'].plot(kind='bar', ax=ax, color=['#d62728', '#2ca02c'])
ax.set_xticklabels(['Auto-renew off', 'Auto-renew on'], rotation=0)
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate by auto-renew status')
for i, v in enumerate(auto_renew_churn['churn_rate']):
    ax.text(i, v + 0.01, f'{v:.1%}', ha='center')
plt.tight_layout()
plt.show()

## Churn by age band

In [ ]:
age_churn = df.groupby('age_band')['is_churn'].agg(['count', 'mean']).rename(columns={'mean': 'churn_rate'})
age_churn = age_churn.reindex(['<20', '20-29', '30-39', '40-49', '50-59', '60+', 'unknown'])
age_churn

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
age_churn['churn_rate'].plot(kind='bar', ax=ax, color='#1f77b4')
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate by age band')
ax.set_xticklabels(age_churn.index, rotation=30)
for i, v in enumerate(age_churn['churn_rate']):
    if not np.isnan(v):
        ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Churn by engagement-drop quintile (last 15 days)

In [ ]:
drop = df.dropna(subset=['engagement_drop_w15']).copy()
if drop.empty or drop['engagement_drop_w15'].nunique() < 2:
    quintile_churn = pd.DataFrame({'count': [0], 'churn_rate': [np.nan]}, index=['insufficient_variation'])
else:
    drop['drop_quintile'] = pd.qcut(
        drop['engagement_drop_w15'],
        5,
        labels=['Q1 worst', 'Q2', 'Q3', 'Q4', 'Q5 best'],
        duplicates='drop',
    )
    quintile_churn = drop.groupby('drop_quintile', observed=True)['is_churn'].agg(['count', 'mean']).rename(columns={'mean': 'churn_rate'})
quintile_churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
quintile_churn['churn_rate'].plot(kind='bar', ax=ax, color='#ff7f0e')
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate by 15-day engagement-drop quintile')
ax.set_xticklabels(quintile_churn.index, rotation=15)
for i, v in enumerate(quintile_churn['churn_rate']):
    if pd.notna(v):
        ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Churn by discount band

In [ ]:
df['discount_band'] = pd.cut(df['avg_discount_rate'].fillna(0),
                              bins=[-0.01, 0.05, 0.20, 0.40, 1.0],
                              labels=['none', 'low', 'medium', 'heavy'])
discount_churn = df.groupby('discount_band', observed=True)['is_churn'].agg(['count', 'mean']).rename(columns={'mean': 'churn_rate'})
discount_churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
discount_churn['churn_rate'].plot(kind='bar', ax=ax, color='#9467bd')
ax.set_ylabel('Churn rate')
ax.set_title('Churn rate by average discount band')
ax.set_xticklabels(discount_churn.index, rotation=0)
for i, v in enumerate(discount_churn['churn_rate']):
    ax.text(i, v + 0.005, f'{v:.1%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## Listening intensity vs churn

In [ ]:
listen_summary = df.groupby('is_churn')[['active_days_w30', 'listen_seconds_w30', 'unique_songs_w30']].mean().T
listen_summary.columns = ['retained', 'churned']
listen_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
active_box = df[['is_churn', 'active_days_w30']].copy()
active_box['Group'] = active_box['is_churn'].map({0: 'Retained', 1: 'Churned'})
sns.boxplot(data=active_box, x='Group', y='active_days_w30', ax=ax, showfliers=False)
ax.set_title('Active listening days (last 30) by churn status')
plt.tight_layout()
plt.show()

## Headline insights

Read the printed numbers above and update the case-study PDF placeholders accordingly:

1. Auto-renew off vs on — ratio of churn rates.
2. Engagement-drop quintile — Q1 worst vs Q5 best.
3. Discount-heavy users churn rate.

These three insights drive the recommendations in the dashboard.